# SKTECH Face Recognition Training and Monitoring

This notebook monitors the local SKTECH embedding-based face recognition workflow for panel presentation and documentation. It evaluates dataset quality, identity prototypes, threshold calibration, verification metrics, and inference speed.

## Purpose

SKTECH does not train dlib weights from scratch. The workflow uses the same `face_recognition` embedding pipeline as the FastAPI service, builds local identity prototypes from consented images, evaluates verification thresholds, and measures processing speed. Raw images and biometric-derived artifacts stay local.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'ai-training').exists() and (ROOT.parent / 'ai-training').exists():
    ROOT = ROOT.parent
WORKSPACE = ROOT / 'ai-training'
DATA = WORKSPACE / 'data' / 'raw'
REPORT = WORKSPACE / 'reports' / 'face_evaluation_summary.json'
SWEEP = WORKSPACE / 'reports' / 'threshold_sweep.csv'
print(f'Workspace: {WORKSPACE}')
print(f'Dataset: {DATA}')

## Dataset Overview

Use only consented local images under `ai-training/data/raw/<person_id>/`. If the dataset is empty, the scripts and this notebook show a clear message instead of failing.

In [ ]:
image_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
image_paths = sorted(path for path in DATA.rglob('*') if path.is_file() and path.suffix.lower() in image_extensions) if DATA.exists() else []
if not image_paths:
    print('No local face dataset found. Add consented images to ai-training/data/raw/ to run training and evaluation.')
else:
    overview = pd.Series([path.parent.name for path in image_paths]).value_counts().rename_axis('identity').reset_index(name='images')
    print(f'Identity count: {overview.shape[0]}')
    print(f'Image count: {len(image_paths)}')
    display(overview)

In [ ]:
# Optional preprocessing preview: draw the detected face box on one local sample.
import face_recognition

if not image_paths:
    print("No preprocessing preview available without a local dataset.")
else:
    preview_path = image_paths[0]
    preview_image = face_recognition.load_image_file(preview_path)
    preview_locations = face_recognition.face_locations(preview_image, model="hog")
    plt.figure(figsize=(8, 5))
    plt.imshow(preview_image)
    for top, right, bottom, left in preview_locations:
        plt.gca().add_patch(
            plt.Rectangle(
                (left, top), right - left, bottom - top,
                fill=False, color="lime", linewidth=2,
            )
        )
    plt.title(f"Preprocessing preview: {preview_path.name} ({len(preview_locations)} face(s))")
    plt.axis("off")
    plt.show()

## Embedding Generation and Enrollment Prototype

The training script detects one face per image, generates a 128-dimensional dlib embedding, and averages enrollment samples into one identity prototype. Detection failures are recorded rather than hidden.

In [ ]:
train_script = WORKSPACE / 'scripts' / 'train_face_model.py'
result = subprocess.run([sys.executable, str(train_script), '--data', str(DATA)], cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
prototype_path = WORKSPACE / 'models' / 'face_prototypes.json'
if prototype_path.exists():
    prototype_model = json.loads(prototype_path.read_text(encoding='utf-8'))
    print('Embedding dimension:', prototype_model.get('embeddingDimension'))
    print('Identity prototypes:', len(prototype_model.get('identities', [])))
    print('Successful images:', prototype_model.get('metadata', {}).get('successCount', 0))
    print('Failed images:', prototype_model.get('metadata', {}).get('failedDetectionCount', 0))

## Threshold Sweep and Verification Metrics

Validation samples are compared with identity prototypes across multiple Euclidean-distance thresholds. The report includes accuracy, FAR, FRR, precision, recall, true accepts, false accepts, true rejects, and false rejects.

In [ ]:
evaluate_script = WORKSPACE / 'scripts' / 'evaluate_face_model.py'
result = subprocess.run([sys.executable, str(evaluate_script), '--data', str(DATA)], cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
summary = json.loads(REPORT.read_text(encoding='utf-8')) if REPORT.exists() else {}
sweep = pd.read_csv(SWEEP) if SWEEP.exists() and SWEEP.stat().st_size > 0 else pd.DataFrame()
if sweep.empty:
    print('No threshold metrics available. Add at least two valid images per identity.')
else:
    display(sweep)
    sweep.plot(x='threshold', y=['accuracy', 'far', 'frr'], marker='o', ylim=(0, 1), figsize=(9, 4), title='Verification threshold sweep')
    plt.ylabel('Rate')
    plt.grid(alpha=0.25)
    plt.show()

## Speed Monitoring

The evaluator records preprocessing, face detection, embedding generation, comparison, and total verification time.

In [ ]:
timing = summary.get('timingMs', {})
timing_frame = pd.DataFrame({'stage': list(timing.keys()), 'milliseconds': list(timing.values())})
timing_frame = timing_frame.dropna()
if timing_frame.empty:
    print('No timing metrics available yet.')
else:
    display(timing_frame)
    timing_frame.plot.bar(x='stage', y='milliseconds', legend=False, figsize=(9, 4), title='Average verification timing')
    plt.ylabel('Milliseconds')
    plt.xticks(rotation=25, ha='right')
    plt.tight_layout()
    plt.show()

## Recommendation

The recommended threshold is an evaluation result only. Do not change the production threshold without reviewing FAR/FRR, liveness behavior, operational conditions, and security approval.

In [ ]:
recommended = summary.get('recommendedThreshold')
print('Recommended threshold:', recommended)
print('Optimization notes: resize/compress inputs, keep the service warm, avoid model reloads, and measure p95 latency before changing runtime behavior.')
print('Panel metrics:', summary.get('metrics', {}))

## Panel Explanation

This notebook does not train a large face model from scratch. It trains/evaluates the SKTECH face recognition pipeline by generating embeddings, building identity prototypes, testing thresholds, and measuring verification speed.